# MiniCheck — v1.2 prompt-intervention study

Evaluates all 1,393 frozen summary sentences from 54 blinded summaries against their complete source privacy policy. MiniCheck is a **binary faithfulness evaluator**: `1 = predicted supported`, `0 = predicted unsupported`. It does not measure omission/coverage.

Use **Runtime → Change runtime type → T4 GPU**. Upload only `MiniCheck_V12_inputs.zip`. Progress is saved in Google Drive after every batch, so the run can resume after disconnection.


In [ ]:
%pip install -q "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


In [ ]:
from google.colab import files, drive
from pathlib import Path
import io, zipfile, pandas as pd

uploaded = files.upload()
name = 'MiniCheck_V12_inputs.zip'
if name not in uploaded:
    raise FileNotFoundError(f'Upload exactly {name}')
INPUT_DIR = Path('/content/minicheck_v12_inputs')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
    z.extractall(INPUT_DIR)
claims = pd.read_csv(INPUT_DIR / 'v12_minicheck_statements.csv', dtype=str).fillna('')
assert len(claims) == 1393
assert claims.statement_id.is_unique
print('Validated statements:', len(claims), 'Summaries:', claims.blind_id.nunique())


In [ ]:
import torch
from minicheck.minicheck import MiniCheck

MODEL_ID = 'flan-t5-large'
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
scorer = MiniCheck(model_name=MODEL_ID, cache_dir='/content/minicheck_ckpts')


In [ ]:
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/MSc_AI_MiniCheck_V12')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = DRIVE_DIR / 'minicheck_v12_statement_scores.csv'

sources = {p.stem: p.read_text(encoding='utf-8').strip() for p in (INPUT_DIR / 'sources').glob('*.txt')}
if OUTPUT.exists():
    done = pd.read_csv(OUTPUT, dtype=str).fillna('')
else:
    done = pd.DataFrame(columns=['blind_id','policy_id','statement_id','statement_text','minicheck_model','predicted_supported','probability','status','error'])
done_ids = set(done.loc[done.status.eq('success'), 'statement_id'])
pending = claims.loc[~claims.statement_id.isin(done_ids)].copy()
print('Already complete:', len(done_ids), 'Pending:', len(pending))


In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 16  # Reduce to 8 if Colab runs out of GPU memory; do not exceed 32 without testing.
rows = done.to_dict('records')
for start in range(0, len(pending), BATCH_SIZE):
    batch = pending.iloc[start:start+BATCH_SIZE]
    try:
        labels, probabilities, _, _ = scorer.score(
            docs=[sources[p] for p in batch.policy_id],
            claims=batch.statement_text.tolist(),
        )
        for (_, item), label, probability in zip(batch.iterrows(), labels, probabilities):
            rows.append({**item.to_dict(), 'minicheck_model': MODEL_ID,
                         'predicted_supported': int(label), 'probability': float(probability),
                         'status': 'success', 'error': ''})
    except Exception as exc:
        for _, item in batch.iterrows():
            rows.append({**item.to_dict(), 'minicheck_model': MODEL_ID,
                         'predicted_supported': '', 'probability': '', 'status': 'failed',
                         'error': f'{type(exc).__name__}: {exc}'})
    current = pd.DataFrame(rows).drop_duplicates('statement_id', keep='last')
    current.to_csv(OUTPUT, index=False)
    print(f'Saved {min(start+BATCH_SIZE, len(pending))}/{len(pending)} pending statements')
print('Run finished:', OUTPUT)


In [ ]:
results = pd.read_csv(OUTPUT, dtype=str).fillna('')
success = results[results.status.eq('success')].copy()
assert success.statement_id.is_unique
print('Successful:', len(success), '/ 1393')
print('Summaries represented:', success.blind_id.nunique(), '/ 54')
print(success.predicted_supported.value_counts())
if len(success) == 1393:
    final_path = DRIVE_DIR / 'minicheck_v12_statement_scores_FINAL.csv'
    success.sort_values(['blind_id','statement_id']).to_csv(final_path, index=False)
    files.download(str(final_path))
else:
    print('Not complete yet. Re-run the notebook later; it will resume from Google Drive.')


## Return the result

When the final validation reports **1,393 / 1,393 successful** and **54 / 54 summaries**, place the downloaded `minicheck_v12_statement_scores_FINAL.csv` in:

`pilot_study/data/v2_prompt_intervention_v1_2/evaluations/minicheck/`

Do not translate MiniCheck's binary output into the four-category Gemini/human labels.
